<a href="https://colab.research.google.com/github/JinyongPark72/Login/blob/master/drive_cb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip3 install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0970a497bc147ab990aac528ee368170f57fec3b2f3393e0ab6a8d3493dbb4d4
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import whisper
from openai import OpenAI
import json
import os

# 🔑 OpenAI API (환경변수 사용)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 🧠 상태 저장
state = {
    "출발지": None,
    "도착지": None,
    "상태": "입력중"  # 입력중 / 확인중 / 완료
}

# ✅ 응답 판단
YES = ["네", "예", "맞", "응", "그래"]
NO = ["아니", "틀", "아냐"]

def is_yes(text):
    return any(word in text for word in YES)

def is_no(text):
    return any(word in text for word in NO)

# 🎤 Whisper 모델 (한 번만 로딩)
model = whisper.load_model("small")

# 1. 음성 → 텍스트
def speech_to_text(audio_file):
    result = model.transcribe(audio_file, language='ko')
    return result["text"]

# 2. 조사 제거
def clean_location(text):
    particles = ["에서", "에", "로", "으로"]
    for p in particles:
        if text.endswith(p):
            text = text[:-len(p)]
    return text.strip()

# 3. 유효한 장소인지 확인
def is_valid_location(text):
    invalid_words = ["싶어요", "가고", "해주세요", "부탁", "지금", "현재 위치", "여기", "집"]

    if not text:
        return False

    return not any(word in text for word in invalid_words)

def is_new_request(text):
    keywords = ["출발", "까지", "가고 싶", "가주세요", "목적지"]
    return any(word in text for word in keywords)

# 4. AI로 출발/도착 추출 (🔥 안정화 버전)
def extract_with_ai(text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": """
                    출발지와 도착지를 JSON으로만 반환해.

                    규칙:
                    - 출발지가 명확하지 않으면 null로 반환
                    - 절대 임의로 '현재 위치', '여기' 같은 값 넣지 마
                    - 반드시 JSON 형식만 출력

                    예:
                    {"출발지": null, "도착지": "잠실"}
                    """
                },
                {"role": "user", "content": text}
            ]
        )

        content = response.choices[0].message.content.strip()

        # 🔥 GPT 코드블록 제거
        content = content.replace("```json", "").replace("```", "").strip()

        data = json.loads(content)

        return data.get("출발지"), data.get("도착지")

    except Exception as e:
        print("GPT 파싱 오류:", e)
        return None, None

# 5. 대화 처리
def process(text):
    global state

    print("🧪 입력:", text)
    print("🧪 현재 상태:", state)

    # 🔥 공백 제거 (필수)
    text = text.strip().replace("  ", " ")

    # 🔥 새로운 요청이면 상태 초기화
    if state["상태"] != "입력중" and is_new_request(text):
        state["출발지"] = None
        state["도착지"] = None
        state["상태"] = "입력중"

    # 🔥 이미 완료 상태
    if state["상태"] == "완료":
        print("🚗 이미 접수 완료")
        return

    # 🔥 확인 단계
    if state["상태"] == "확인중":
        if is_yes(text):
            state["상태"] = "완료"
            print("🚗 접수 완료!")
            print("🧪 상태 변경 → 완료")
            print(f"출발지: {state['출발지']}")
            print(f"도착지: {state['도착지']}")
            return

        elif is_no(text):
            state["출발지"] = None
            state["도착지"] = None
            state["상태"] = "입력중"
            print("👉 다시 말씀해주세요")
            return

    # 🔥 입력 단계
    start, end = extract_with_ai(text)
    print("🧪 GPT 결과:", start, end)

    if start is None and end is None:
        print("👉 다시 한 번 말씀해주세요")
        return

    # 🔥 새 입력인데 출발지 없으면 기존값 제거
    if start is not None and is_valid_location(start):
        state["출발지"] = clean_location(start)

    if end is not None and is_valid_location(end):
        state["도착지"] = clean_location(end)

    # 🔥 둘 다 있고 아직 입력중일 때만 확인 질문
    if state["출발지"] and state["도착지"] and state["상태"] == "입력중":
        print(f"👉 {state['출발지']} → {state['도착지']} 맞으세요?")
        state["상태"] = "확인중"
        print("🧪 상태 변경 → 확인중")
        return

    # 🔥 부족한 정보 질문
    if not state["출발지"]:
        print("👉 출발지 말씀해주세요")
    elif not state["도착지"]:
        print("👉 도착지 말씀해주세요")

# 6. 실행
if __name__ == "__main__":
    print("🎤 음성을 입력하세요 (파일 입력 방식)")

    while True:
        audio_file = input("파일 이름 입력: ")

        try:
            text = speech_to_text(audio_file)
        except Exception as e:
            print("❌ 음성 인식 실패:", e)
            continue

        print(f"📝 인식된 문장: {text}")

        process(text)

🎤 음성을 입력하세요 (파일 입력 방식)
파일 이름 입력: test1.m4a


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 인식된 문장:  남에서 잠실 갑니다.
🧪 입력:  남에서 잠실 갑니다.
🧪 현재 상태: {'출발지': None, '도착지': None, '상태': '입력중'}
🧪 GPT 결과: 남 잠실
👉 남 → 잠실 맞으세요?
🧪 상태 변경 → 확인중
파일 이름 입력: test2.m4a


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 인식된 문장:  잠실 가는데 강남에서 출발
🧪 입력:  잠실 가는데 강남에서 출발
🧪 현재 상태: {'출발지': '남', '도착지': '잠실', '상태': '확인중'}
🧪 GPT 결과: 강남 잠실
👉 강남 → 잠실 맞으세요?
🧪 상태 변경 → 확인중
파일 이름 입력: test3.m4a


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 인식된 문장:  잠실 가고 싶어요
🧪 입력:  잠실 가고 싶어요
🧪 현재 상태: {'출발지': '강남', '도착지': '잠실', '상태': '확인중'}
🧪 GPT 결과: None 잠실
👉 출발지 말씀해주세요
파일 이름 입력: test4.m4a


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 인식된 문장:  강남
🧪 입력:  강남
🧪 현재 상태: {'출발지': None, '도착지': '잠실', '상태': '입력중'}
🧪 GPT 결과: 강남 None
👉 강남 → 잠실 맞으세요?
🧪 상태 변경 → 확인중
